# 🔄 Iterative Deepening Search (IDS) - 8-Puzzle

File này triển khai giải thuật **Iterative Deepening Search (IDS)** để giải bài toán 8-Puzzle với hai cách tiếp cận:
1. **Cách 1**: Kiểm tra mục tiêu (Goal-test) khi **lấy nút ra** khỏi stack duyệt độ sâu.
2. **Cách 2**: Kiểm tra mục tiêu (Early Goal Check) ngay khi **sinh nút con**.


## 🛠️ 1. Khai báo thư viện & Định nghĩa môi trường

In [ ]:
import copy
import time

# Trạng thái ban đầu & Mục tiêu
initial_state = [
    [1, 2, 3],
    [4, 0, 6],
    [7, 5, 8]
]

goal_state = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

def state_to_tuple(state):
    return tuple(tuple(row) for row in state)

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    return -1, -1

def generate_children(state):
    children = []
    x, y = find_zero(state)
    moves = [
        ("LÊN", -1, 0),
        ("XUỐNG", 1, 0),
        ("TRÁI", 0, -1),
        ("PHẢI", 0, 1)
    ]
    for move_name, dx, dy in moves:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            new_state = copy.deepcopy(state)
            new_state[x][y], new_state[nx][ny] = new_state[nx][ny], new_state[x][y]
            children.append((move_name, new_state))
    return children


## 🚶‍♂️ 2. Triển khai Depth-Limited Search & IDS - Cách 1

In [ ]:
def dls_way_1(state, goal, limit, path, visited_path, stats):
    stats['steps'] += 1
    
    # Goal test khi duyệt nút
    if state == goal:
        return path, True
        
    if limit <= 0:
        return "cutoff", False
        
    cutoff_occurred = False
    state_tuple = state_to_tuple(state)
    visited_path.add(state_tuple)
    
    for move, child in generate_children(state):
        child_tuple = state_to_tuple(child)
        if child_tuple not in visited_path:
            res, found = dls_way_1(child, goal, limit - 1, path + [move], visited_path, stats)
            if found:
                return res, True
            if res == "cutoff":
                cutoff_occurred = True
                
    visited_path.remove(state_tuple)
    return "cutoff" if cutoff_occurred else None, False

def ids_way_1(init, goal):
    max_depth = 50
    total_steps = 0
    
    for depth in range(max_depth):
        stats = {'steps': 0}
        visited = set()
        res, found = dls_way_1(init, goal, depth, [], visited, stats)
        total_steps += stats['steps']
        
        if found:
            return res, depth, total_steps
        if res != "cutoff":
            break
            
    return None, -1, total_steps


## ⚡ 3. Triển khai Depth-Limited Search & IDS - Cách 2

In [ ]:
def dls_way_2(state, goal, limit, path, visited_path, stats):
    stats['steps'] += 1
    
    if limit <= 0:
        return "cutoff", False
        
    cutoff_occurred = False
    state_tuple = state_to_tuple(state)
    visited_path.add(state_tuple)
    
    for move, child in generate_children(state):
        # Early check mục tiêu
        if child == goal:
            return path + [move], True
            
        child_tuple = state_to_tuple(child)
        if child_tuple not in visited_path:
            res, found = dls_way_2(child, goal, limit - 1, path + [move], visited_path, stats)
            if found:
                return res, True
            if res == "cutoff":
                cutoff_occurred = True
                
    visited_path.remove(state_tuple)
    return "cutoff" if cutoff_occurred else None, False

def ids_way_2(init, goal):
    if init == goal:
        return [], 0, 0
        
    max_depth = 50
    total_steps = 0
    
    for depth in range(max_depth):
        stats = {'steps': 0}
        visited = set()
        res, found = dls_way_2(init, goal, depth, [], visited, stats)
        total_steps += stats['steps']
        
        if found:
            return res, depth, total_steps
        if res != "cutoff":
            break
            
    return None, -1, total_steps


## 📊 4. So sánh kết quả chạy thử

In [ ]:
print("=== CHẠY IDS CÁCH 1 (GOAL-TEST ON VISIT) ===")
start = time.perf_counter()
path1, depth1, steps1 = ids_way_1(initial_state, goal_state)
end = time.perf_counter()
print(f"Đường đi: {path1}")
print(f"Tìm thấy ở độ sâu: {depth1}")
print(f"Tổng số lần gọi nút: {steps1}")
print(f"Thời gian chạy: {(end - start)*1000:.4f} ms\n")

print("=== CHẠY IDS CÁCH 2 (EARLY GOAL CHECK) ===")
start = time.perf_counter()
path2, depth2, steps2 = ids_way_2(initial_state, goal_state)
end = time.perf_counter()
print(f"Đường đi: {path2}")
print(f"Tìm thấy ở độ sâu: {depth2}")
print(f"Tổng số lần gọi nút: {steps2}")
print(f"Thời gian chạy: {(end - start)*1000:.4f} ms\n")


## 🎨 5. Trực quan hóa kết quả bằng Pygame (Cửa sổ GUI)

Chạy cell dưới đây để khởi chạy cửa sổ Pygame hiển thị quá trình giải bài toán.
- Nhấn **SPACE** để Tạm dừng/Tiếp tục chạy tự động.
- Nhấn phím **MŨI TÊN TRÁI / PHẢI** để tua lại / chuyển bước thủ công.
- Nhấn **R** để khởi động lại quá trình từ đầu.

🚀 **Mẹo:** Bạn có thể chạy ứng dụng trực quan tương tác hoàn chỉnh (cho phép tự chọn thuật toán, đổi bài toán, ngẫu nhiên hóa và xem bảng so sánh thống kê) bằng cách chạy trực tiếp file [puzzle_visualizer.py](file:///G:/Desktop/tri_tue_nhan_tao/Search_Algorithm/puzzle_visualizer.py) trong terminal: 
```bash
python puzzle_visualizer.py
```


In [ ]:
import sys
sys.path.append('../../')
from utils.common import visualize_puzzle_pygame

# Chạy trực quan hóa Pygame
visualize_puzzle_pygame(initial_state, path2, delay=1.0)
